## Creating an Wizard of Oz expert assistant

Here we will demonstrate how to create a simple AI assistant to answer question about the book **"The Wizard of Oz"**.

In order to do so, we will use **Mosaic AI Foundation Models** to generate our answers without the hustles of training and serving large language models.

Also, we will use **Chroma** vector database to provide further context regarding the contents of the book to our GenAI models. This will help us achieve higher accuracy in our responses and avoid hallucinations.

Then we create our assistant by chaining this components with additional instructions using **Langchain**.

Finally, we will create a small **Gradio** app to easily interact with our assistant.

Let's get started!

### Setup

Before developing our assistant, let's install and import all additional libraries not yet present in our cluster's **Databricks Runtime**.

In [0]:
%pip install -U chromadb
dbutils.library.restartPython()

In [0]:
from langchain.document_loaders import GutenbergLoader
from langchain.vectorstores import Chroma
from langchain.embeddings import DatabricksEmbeddings
from langchain.chat_models import ChatDatabricks
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from transformers import OpenAIGPTTokenizer
import chromadb

### Load data

Now, let's download the full text of the book.

In [0]:
full_text = GutenbergLoader("https://www.gutenberg.org/files/55/55-0.txt").load() # Wizard of Oz

### Chunking

Then we need to split this text into smaller pieces, so we can search for required information more easily later on.

Here we will create 500-characters-long chunks with an overlap of 50 characters.

In [0]:
max_chunk_size = 500

tokenizer = OpenAIGPTTokenizer.from_pretrained("openai-gpt")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=max_chunk_size, chunk_overlap=50)
chunks = text_splitter.split_documents(full_text)

### Indexing documents

Once we have our chunks, we can index these into our **Chroma** vector database, so it can be used to retrieve relevant pieces of the book for a particular query.

Vector databases require all textual data to be converted into embeddings, so we will use **Databricks Foundation Models** to do so. Here we have selected **BGE-large-en**.

In [0]:
embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")
vectordb = Chroma.from_documents(chunks, embeddings)

Now, let's see if our vector datbase is working fine.

In [0]:
vectordb.similarity_search("Did Tin Woodman get a heart?")

### Chaining

Now that we have a vector database, we will create a chain to orchestrate all the steps our assistant need to follow in order to answer a question.

Additionally, we will create a prompt template to instruct our model on which kind of questions it can answer and how it should respond. The idea here is to create some guardrails for our assistant.

Our chain will look like this:
1. Retrieve the most relevant document from the vector database
2. Prepare the prompt by adding the retrieved document as context
3. Generate the final answer by querying the Llama 3.3 70b **Databricks Foundation Model**

In [0]:
# Select our Databricks Foundation Model
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.1)

# Define our prompt
template = '''
You are an Wizard of Oz expert assistant. 
You only answer questions about the book The Wizard of Oz. If the question is not about the book, politely answer that you are not allowed to answer this type of question.
If you don't know the anwer, just say that you don't know. Don't try to make up an answer.
Use the following pieces of context to answer the question at the end.

Context: {context}

Question: {question}

Answer:
'''

# Create a prompt template
prompt = PromptTemplate(
    template=template, 
    input_variables=[
        'context', 
        'question',
    ]
)

# Define a function to join the retrieve documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain all steps together
chain = (
    {
        "context": vectordb.as_retriever(search_kwargs={"k":1}) | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

Now, let's check how our assistant will perform against a few questions.

Here we can also use **MLflow Trace UI** to understand what is happening in each step of the assistant's processing. For example, it makes it possible for us to see which documents are being retrieved, the formatted prompt, inputs and outputs generated, so we can debug eventual errors we may find or identify processing bottlenecks.

In [0]:
query = "Did Tin Woodman get a heart?"
chain.invoke(query)

In [0]:
query = "What is Databricks?"
chain.invoke(query)

### Log Model

The next step is to log our model to MLflow in order to keep track of all experiments that we have built and make sure we are able to reproduce them in the future.

In [0]:
import mlflow

# Define a function to load the retriever
def load_retriever(persist_dir):
  max_chunk_size = 500
  tokenizer = OpenAIGPTTokenizer.from_pretrained("openai-gpt")
  text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=max_chunk_size, chunk_overlap=50)
  chunks = text_splitter.split_documents(full_text)
  embeddings = DatabricksEmbeddings(endpoint="databricks-bge-large-en")
  vectordb = Chroma.from_documents(chunks, embeddings)
  return vectordb.as_retriever()

with mlflow.start_run() as run:
  mlflow.langchain.log_model(
    lc_model=chain,
    artifact_path="model",
    loader_fn=load_retriever,
    input_example=['Did Tin Woodman get a heart?']
  )

### Register model

When we are satisfied with our assistant, we can register our champion model to MLflow in order to version control our production models and govern their lifecycle.

In [0]:
model_name = 'oz_expert'

registered_model = mlflow.register_model(
  model_uri=f'runs:/{run.info.run_id}/model',
  name=model_name
)

### Deploy to Model Serving

Next we will use **Mosaic AI Model Serving** to create a REST API to expose our models to be consumed on external applications, like customer portals, ERPs and CRMs.

In [0]:
# We used the following code to store our authentication information securely using Databricks Secrets

# from databricks.sdk import WorkspaceClient
#
# w = WorkspaceClient()
#
# w.secrets.create_scope('vr')
# w.secrets.put_secret(scope='vr', key='oz_expert_host', string_value='https://interviews.cloud.databricks.com')
# w.secrets.put_secret(scope='vr', key='oz_expert_pat', string_value='dapi...')

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedModelInput, ServedModelInputWorkloadSize

# Create a Databricks SDK Client
w = WorkspaceClient()

# Configure Model Serving endpoint settings
endpoint_config = EndpointCoreConfigInput(
    name=model_name,
    served_models=[
        ServedModelInput(
            model_name=model_name,
            model_version=registered_model.version,
            workload_size=ServedModelInputWorkloadSize.SMALL,
            scale_to_zero_enabled=True,
            environment_vars={
                "DATABRICKS_HOST": "{{secrets/vr/oz_expert_host}}",
                "DATABRICKS_TOKEN": "{{secrets/vr/oz_expert_pat}}"
            }
        )
    ]
)

# Create Model Serving endpoint
w.serving_endpoints.create_and_wait(name=model_name, config=endpoint_config)

Now, let's test our endpoint

In [0]:
w.serving_endpoints.query(name=model_name, inputs=['Did Tin Woodman get a heart?']).predictions[0]

### App

Finally, let's create our **Gradio** app to showcase our model in a closer fashion to what it would look like in the real world.

Here we use the `ChatInterface` template to quickly create a full-fledged chatbot UI with very little effort. Basically, we only need to define a function to answer the questions using our Model Serving endpoint and customize a few parameters, like the title, user instructions and example questions.

Let's see it in practice!

In [0]:
%pip install -U gradio
dbutils.library.restartPython()

In [0]:
import gradio as gr
from databricks.sdk import WorkspaceClient

# Create a Databricks SDK Client
w = WorkspaceClient()

# Define function to answer questions using the assistant
def respond(message, history):
    if len(message.strip()) == 0:
        return "ERROR the question should not be empty"
    try:
        r = w.serving_endpoints.query(name='oz_expert', inputs=[message]).predictions[0]
    except Exception as error:
        return f"ERROR: {error}"
    return r

# Configure Gradio app
demo = gr.ChatInterface(
    respond,
    title="Wizard of Oz expert assistant",
    description="Hi, I am an expert on The Wizard of Oz.<br>Feel free to ask me any questions about the book!<br>I'll be happy to help!",
    examples=[
        ["Did Tin Woodman get a heart?"],
        ["Where does Dorothy live?"],
        ["What is the end of the end of the Wicked Witch?"],
        ["What is Databricks?"]
    ],
    cache_examples=False,
)

# Launch Gradio app
demo.launch(share=True, debug=True)